# Gamma Exposure (GEX): Dealer Hedging and Market Structure

This notebook reconstructs the analytical framework behind **Gamma Exposure (GEX)** — the aggregate gamma positioning of options market makers and its mechanical impact on the underlying market.

The framework was originally developed and published by [SqueezeMetrics](https://squeezemetrics.com/monitor/docs). We derive every formula from first principles, implement the calculations, and build interactive visualizations to develop intuition for how dealer hedging flows create regimes of suppressed or amplified volatility.

**Prerequisites:** Familiarity with Black-Scholes, delta hedging, and the options Greeks (see `options_greeks.ipynb`).

---

## Table of Contents

1. [The Central Insight: Dealers as Mechanical Traders](#1-central-insight)
2. [Market Microstructure: Who Holds What](#2-microstructure)
3. [From Delta Hedging to Gamma Flows](#3-delta-hedging-to-gamma-flows)
4. [Deriving the GEX Formula](#4-deriving-gex)
5. [Computing GEX From an Options Chain](#5-computing-gex)
6. [The Gamma Flip Point](#6-gamma-flip)
7. [GEX Regimes: Positive vs Negative Gamma](#7-gex-regimes)
8. [Simulating Market Impact of Dealer Hedging](#8-market-impact)
9. [GEX and Realized Volatility](#9-gex-and-rvol)
10. [Practical Trading Applications](#10-practical)
11. [Limitations and Extensions](#11-limitations)

In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
})

LONG_GAMMA_COLOR = '#4CAF50'   # green — stabilizing
SHORT_GAMMA_COLOR = '#F44336'  # red — destabilizing
CALL_COLOR = '#2196F3'
PUT_COLOR = '#E91E63'
NEUTRAL_COLOR = '#9E9E9E'

In [ ]:
class BlackScholes:
    """Black-Scholes pricing and Greeks for European options."""

    def __init__(self, S, K, T, r, sigma):
        self.S = np.asarray(S, dtype=float)
        self.K = np.asarray(K, dtype=float)
        self.T = np.asarray(T, dtype=float)
        self.r = np.asarray(r, dtype=float)
        self.sigma = np.asarray(sigma, dtype=float)
        self._compute_d()

    def _compute_d(self):
        sqrt_T = np.sqrt(np.maximum(self.T, 1e-10))
        self.d1 = (np.log(self.S / self.K) + (self.r + 0.5 * self.sigma**2) * self.T) / (self.sigma * sqrt_T)
        self.d2 = self.d1 - self.sigma * sqrt_T

    def call_price(self):
        return self.S * norm.cdf(self.d1) - self.K * np.exp(-self.r * self.T) * norm.cdf(self.d2)

    def put_price(self):
        return self.K * np.exp(-self.r * self.T) * norm.cdf(-self.d2) - self.S * norm.cdf(-self.d1)

    def call_delta(self):
        return norm.cdf(self.d1)

    def put_delta(self):
        return norm.cdf(self.d1) - 1.0

    def gamma(self):
        sqrt_T = np.sqrt(np.maximum(self.T, 1e-10))
        return norm.pdf(self.d1) / (self.S * self.sigma * sqrt_T)

    def dollar_gamma(self):
        """Gamma in dollar terms: shares traded per $1 move × $1 = Γ × S."""
        return self.gamma() * self.S

    def call_theta(self):
        sqrt_T = np.sqrt(np.maximum(self.T, 1e-10))
        term1 = -(self.S * norm.pdf(self.d1) * self.sigma) / (2 * sqrt_T)
        term2 = -self.r * self.K * np.exp(-self.r * self.T) * norm.cdf(self.d2)
        return (term1 + term2) / 365.0

    def vega(self):
        sqrt_T = np.sqrt(np.maximum(self.T, 1e-10))
        return self.S * norm.pdf(self.d1) * sqrt_T / 100.0

---
## 1. The Central Insight: Dealers as Mechanical Traders <a id='1-central-insight'></a>

Options are traded between two broad groups:

- **Customers** (hedge funds, institutions, retail): trade options to express views or hedge portfolios. They choose *when* and *what* to trade.
- **Dealers** (market makers): provide liquidity. They take the other side of customer trades and then **delta-hedge** to remain market-neutral.

The key insight of the GEX framework:

> Because dealers must continuously delta-hedge, their aggregate **gamma exposure** creates **predictable, mechanical buying and selling pressure** on the underlying asset.

This pressure isn't discretionary — it's a *consequence of risk management*. When gamma is large enough relative to natural trading volume, it becomes the dominant force setting the market's volatility regime.

### The Two Regimes

| Dealer Gamma | Hedging Behavior | Market Effect | Volatility |
|:---:|:---:|:---:|:---:|
| **Long gamma** (Γ > 0) | Sell rallies, buy dips | Dampening | Suppressed |
| **Short gamma** (Γ < 0) | Buy rallies, sell dips | Amplifying | Elevated |

In [ ]:
# Visualize the two gamma regimes with a simple diagram
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

S_moves = np.linspace(-5, 5, 200)

# Long gamma dealer
ax = axes[0]
gamma_val = 0.03
hedge_flow_long = -gamma_val * S_moves * 500  # negative = sell when S rises
ax.plot(S_moves, hedge_flow_long, color=LONG_GAMMA_COLOR, linewidth=3)
ax.fill_between(S_moves, 0, hedge_flow_long, alpha=0.15, color=LONG_GAMMA_COLOR)
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('Stock Price Change ($)', fontsize=12)
ax.set_ylabel('Dealer Hedge Flow (shares)', fontsize=12)
ax.set_title('LONG GAMMA DEALER\n(Stabilizing)', fontsize=14, fontweight='bold', color=LONG_GAMMA_COLOR)

ax.annotate('Price rises →\nDealer SELLS', xy=(3, hedge_flow_long[150]),
            xytext=(3.5, 20), fontsize=11, color=LONG_GAMMA_COLOR, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=LONG_GAMMA_COLOR))
ax.annotate('Price falls →\nDealer BUYS', xy=(-3, -hedge_flow_long[50]),
            xytext=(-4.5, -20), fontsize=11, color=LONG_GAMMA_COLOR, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=LONG_GAMMA_COLOR))

# Short gamma dealer
ax = axes[1]
hedge_flow_short = gamma_val * S_moves * 500  # positive = buy when S rises
ax.plot(S_moves, hedge_flow_short, color=SHORT_GAMMA_COLOR, linewidth=3)
ax.fill_between(S_moves, 0, hedge_flow_short, alpha=0.15, color=SHORT_GAMMA_COLOR)
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('Stock Price Change ($)', fontsize=12)
ax.set_ylabel('Dealer Hedge Flow (shares)', fontsize=12)
ax.set_title('SHORT GAMMA DEALER\n(Destabilizing)', fontsize=14, fontweight='bold', color=SHORT_GAMMA_COLOR)

ax.annotate('Price rises →\nDealer BUYS', xy=(3, hedge_flow_short[150]),
            xytext=(1, 60), fontsize=11, color=SHORT_GAMMA_COLOR, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=SHORT_GAMMA_COLOR))
ax.annotate('Price falls →\nDealer SELLS', xy=(-3, hedge_flow_short[50]),
            xytext=(-2, -60), fontsize=11, color=SHORT_GAMMA_COLOR, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=SHORT_GAMMA_COLOR))

plt.suptitle('The Two Gamma Regimes: How Dealer Hedging Affects Markets',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 2. Market Microstructure: Who Holds What <a id='2-microstructure'></a>

To compute dealer gamma, we need to know the **sign** of the dealer's position at each strike. We can't observe this directly from public data, so the GEX framework makes a simplifying assumption based on dominant market flows:

### The Standard Positioning Assumption

| Option Type | Customer Tendency | Dealer Position | Dealer Gamma Sign |
|:-----------:|:-----------------:|:---------------:|:-----------------:|
| **Calls** | Net *seller* (covered calls, overwriting) | Net **long** | **Positive** (+Γ) |
| **Puts** | Net *buyer* (portfolio protection, hedging) | Net **short** | **Negative** (−Γ) |

### Why This Assumption?

**Calls:** The largest systematic options flow in equities is institutional **covered call writing** — funds sell upside calls against their stock positions to generate income. This makes dealers net *long* calls at most strikes.

**Puts:** The second-largest systematic flow is institutional **put buying** for downside protection. Portfolio managers buy puts as insurance. Dealers end up net *short* these puts.

This is a simplification — at some strikes, customer flows go the other way. More sophisticated versions of GEX use actual volume/OI data and transaction-level analysis to infer dealer positioning. But this assumption captures the dominant structural pattern.

### The Consequences

- **Long calls** give dealers **positive gamma** → they sell rallies, buy dips → **stabilizing**.
- **Short puts** give dealers **negative gamma** → they sell drops, buy rallies... wait, that sounds stabilizing too?

Let's be precise about the mechanics.

---
## 3. From Delta Hedging to Gamma Flows <a id='3-delta-hedging-to-gamma-flows'></a>

### Single-Option Hedging Mechanics

Consider a dealer who is **long 1 call** at strike $K$:

- Position delta: $+\Delta_C = +N(d_1) > 0$
- To be delta-neutral, the dealer **shorts** $N(d_1) \times 100$ shares of stock.
- Position gamma: $+\Gamma > 0$

When $S$ rises by \$1:
- New delta = $N(d_1) + \Gamma$ → higher delta → dealer's hedge needs *more* short stock
- Dealer **sells** $\Gamma \times 100$ additional shares to rebalance
- **Selling into a rising market → stabilizing**

When $S$ falls by \$1:
- New delta = $N(d_1) - \Gamma$ → lower delta → dealer's hedge needs *less* short stock
- Dealer **buys** $\Gamma \times 100$ shares back
- **Buying into a falling market → stabilizing**

Now consider a dealer who is **short 1 put** at strike $K$:

- Position delta: $-\Delta_P = -(N(d_1) - 1) = 1 - N(d_1) > 0$
- To be delta-neutral, the dealer **shorts** $(1 - N(d_1)) \times 100$ shares.
- Position gamma: $-\Gamma < 0$

When $S$ falls by \$1:
- Put delta becomes more negative → short put delta increases (more positive)
- Dealer must **sell** more stock to offset the increasing delta
- **Selling into a falling market → destabilizing**

When $S$ rises by \$1:
- Put delta becomes less negative → short put delta decreases
- Dealer **buys** stock back
- **Buying into a rising market → destabilizing**

### Summary of Hedge Flows

The number of shares a dealer must **buy** when $S$ rises by \$1:

$$\text{Hedge flow}_{\text{per \$1 up}} = \underbrace{-\Gamma_{\text{call}} \times OI_{\text{call}} \times 100}_{\text{SELL into rally (stabilizing)}} + \underbrace{+\Gamma_{\text{put}} \times OI_{\text{put}} \times 100}_{\text{BUY into rally (destabilizing)}}$$

Negative hedge flow on an up-move means dealers are net selling → stabilizing.

In [ ]:
# Demonstrate hedging mechanics for long call vs short put
S_range = np.linspace(85, 115, 500)
K, T, r, sigma = 100, 30/365, 0.05, 0.20

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# --- Row 1: Long call dealer ---
bs = BlackScholes(S=S_range, K=K, T=T, r=r, sigma=sigma)

# Delta (shares dealer must short to hedge)
axes[0, 0].plot(S_range, bs.call_delta() * 100, color=CALL_COLOR)
axes[0, 0].set_ylabel('Shares to short')
axes[0, 0].set_title('Long Call: Hedge Position')

# Gamma (how hedge changes per $1 move)
axes[0, 1].plot(S_range, -bs.gamma() * 100, color=LONG_GAMMA_COLOR)
axes[0, 1].fill_between(S_range, 0, -bs.gamma() * 100, alpha=0.15, color=LONG_GAMMA_COLOR)
axes[0, 1].set_ylabel('Shares bought per $1 up')
axes[0, 1].set_title('Long Call: Hedge Flow on $1 Up-Move')
axes[0, 1].annotate('Negative = SELLING\n(stabilizing)', xy=(100, -bs.gamma()[250]*100),
                     xytext=(106, -1.5), fontsize=10, color=LONG_GAMMA_COLOR,
                     arrowprops=dict(arrowstyle='->', color=LONG_GAMMA_COLOR))

# Cumulative hedge P&L showing mean reversion effect
np.random.seed(123)
n_paths = 5
n_steps = 30
dt_sim = T / n_steps
for _ in range(n_paths):
    S_path = [100.0]
    for j in range(n_steps):
        dW = np.random.normal(0, np.sqrt(dt_sim))
        S_path.append(S_path[-1] * np.exp((r - 0.5*sigma**2)*dt_sim + sigma*dW))
    axes[0, 2].plot(S_path, alpha=0.6, linewidth=1)
axes[0, 2].axhline(100, color='gray', linestyle=':', linewidth=0.5)
axes[0, 2].set_ylabel('Stock Price ($)')
axes[0, 2].set_title('Long Gamma → Sells high, buys low')

# --- Row 2: Short put dealer ---
# Delta of short put = -(N(d1)-1) = 1-N(d1)
short_put_delta = -bs.put_delta() * 100  # shares to short for hedge
axes[1, 0].plot(S_range, -bs.put_delta() * 100, color=PUT_COLOR)
axes[1, 0].set_ylabel('Shares to short')
axes[1, 0].set_title('Short Put: Hedge Position')

# Gamma of short put = -Γ → when S rises, dealer must buy
axes[1, 1].plot(S_range, bs.gamma() * 100, color=SHORT_GAMMA_COLOR)
axes[1, 1].fill_between(S_range, 0, bs.gamma() * 100, alpha=0.15, color=SHORT_GAMMA_COLOR)
axes[1, 1].set_ylabel('Shares bought per $1 up')
axes[1, 1].set_title('Short Put: Hedge Flow on $1 Up-Move')
axes[1, 1].annotate('Positive = BUYING\n(destabilizing)', xy=(100, bs.gamma()[250]*100),
                     xytext=(106, 1.0), fontsize=10, color=SHORT_GAMMA_COLOR,
                     arrowprops=dict(arrowstyle='->', color=SHORT_GAMMA_COLOR))

# Short gamma: chasing the market
np.random.seed(123)
for _ in range(n_paths):
    S_path = [100.0]
    for j in range(n_steps):
        dW = np.random.normal(0, np.sqrt(dt_sim))
        S_path.append(S_path[-1] * np.exp((r - 0.5*sigma**2)*dt_sim + sigma*dW))
    axes[1, 2].plot(S_path, alpha=0.6, linewidth=1)
axes[1, 2].axhline(100, color='gray', linestyle=':', linewidth=0.5)
axes[1, 2].set_ylabel('Stock Price ($)')
axes[1, 2].set_title('Short Gamma → Buys high, sells low')

for ax in axes.flat:
    ax.set_xlabel('Stock Price ($)' if ax in axes[1] else '')
    ax.axvline(K, color='gray', linewidth=0.5, linestyle=':')

axes[0, 0].text(87, 80, 'LONG CALL\n(+Γ)', fontsize=13, fontweight='bold', color=CALL_COLOR)
axes[1, 0].text(87, 80, 'SHORT PUT\n(−Γ)', fontsize=13, fontweight='bold', color=PUT_COLOR)

plt.suptitle('Dealer Hedging Mechanics: Long Call (+Γ) vs Short Put (−Γ)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Deriving the GEX Formula <a id='4-deriving-gex'></a>

### Step 1: Per-Contract Gamma

Each options contract covers 100 shares. The gamma of one contract at strike $K$ is:

$$\Gamma_{\text{contract}}(K) = \Gamma_{\text{BS}}(K) \times 100$$

This tells us: if $S$ moves by \$1, the contract's delta changes by $\Gamma_{\text{BS}} \times 100$ shares.

### Step 2: Aggregate Across Open Interest

At each strike $K$, there are $OI_{\text{call}}(K)$ call contracts and $OI_{\text{put}}(K)$ put contracts. Using the dealer positioning assumption:

$$\text{Dealer gamma at strike } K = \underbrace{+\Gamma(K) \times OI_{\text{call}}(K) \times 100}_{\text{from long calls}} \underbrace{- \Gamma(K) \times OI_{\text{put}}(K) \times 100}_{\text{from short puts}}$$

### Step 3: Convert to Dollar Terms

Raw gamma is in units of *delta per dollar*. To get the dollar value of shares traded per \$1 move, multiply by $S$. To express per 1% move of the underlying, multiply by $S$ again:

$$\text{GEX}(K) = \Gamma(K) \times OI_{\text{net}}(K) \times 100 \times S^2$$

where $OI_{\text{net}}(K) = OI_{\text{call}}(K) - OI_{\text{put}}(K)$ under the standard sign convention.

### Step 4: Aggregate Across All Strikes

$$\boxed{\text{GEX}_{\text{total}} = \sum_{K} \left[ \Gamma(K) \times OI_{\text{call}}(K) - \Gamma(K) \times OI_{\text{put}}(K) \right] \times 100 \times S^2}$$

### Interpretation

- **GEX > 0:** Call-side gamma dominates → dealers are net long gamma → **stabilizing** regime.
- **GEX < 0:** Put-side gamma dominates → dealers are net short gamma → **destabilizing** regime.
- **Units:** Dollars of stock that dealers must trade per 1% move in the underlying.

In [ ]:
def compute_gex_profile(spot, strikes, call_oi, put_oi, T, r, sigma, spot_range=None):
    """Compute GEX profile across strikes, optionally at different spot levels.

    Returns per-strike GEX (at current spot) and total GEX (at each spot in spot_range).
    """
    if spot_range is None:
        spot_range = np.array([spot])

    # Per-strike GEX at current spot
    bs = BlackScholes(S=spot, K=strikes, T=T, r=r, sigma=sigma)
    gamma_per_strike = bs.gamma()
    gex_call = gamma_per_strike * call_oi * 100 * spot**2
    gex_put = -gamma_per_strike * put_oi * 100 * spot**2
    gex_net = gex_call + gex_put

    # Total GEX as a function of spot (recalculate gamma at each spot level)
    total_gex = np.zeros_like(spot_range, dtype=float)
    for i, s in enumerate(spot_range):
        bs_s = BlackScholes(S=s, K=strikes, T=T, r=r, sigma=sigma)
        g = bs_s.gamma()
        total_gex[i] = np.sum((g * call_oi - g * put_oi) * 100 * s**2)

    return {
        'strikes': strikes,
        'gamma': gamma_per_strike,
        'gex_call': gex_call,
        'gex_put': gex_put,
        'gex_net': gex_net,
        'total_gex_by_spot': total_gex,
        'spot_range': spot_range,
    }

---
## 5. Computing GEX From an Options Chain <a id='5-computing-gex'></a>

Let's build a realistic synthetic options chain for SPX and compute the full GEX profile. We'll model OI patterns that mirror real market structure: heavy put OI below the market (portfolio hedging) and call OI above (covered calls).

In [ ]:
# Synthetic SPX-like options chain
spot = 4500
r = 0.05
T = 30 / 365  # 30 DTE
sigma = 0.18

# Strikes every 25 points, from -15% to +15%
strikes = np.arange(spot * 0.85, spot * 1.15, 25)
moneyness = strikes / spot

# Realistic OI distribution:
# - Large put OI below spot (hedging demand)
# - Peak put OI ~5% OTM
# - Moderate call OI above spot (overwriting)
# - Significant OI at round numbers
np.random.seed(42)

def synthetic_oi(strikes, spot, option_type):
    """Generate realistic OI distribution."""
    m = strikes / spot
    if option_type == 'put':
        # Heavy OI 3-8% below spot, declining further out
        base = 8000 * np.exp(-((m - 0.94)**2) / (2 * 0.03**2))
        base += 3000 * np.exp(-((m - 0.90)**2) / (2 * 0.04**2))
        # ATM puts
        base += 4000 * np.exp(-((m - 1.0)**2) / (2 * 0.01**2))
    else:
        # Moderate OI 2-6% above spot (overwriting)
        base = 5000 * np.exp(-((m - 1.04)**2) / (2 * 0.03**2))
        base += 2000 * np.exp(-((m - 1.08)**2) / (2 * 0.04**2))
        # ATM calls
        base += 3000 * np.exp(-((m - 1.0)**2) / (2 * 0.01**2))

    # Round number bonus (every 100 points gets extra OI)
    round_bonus = np.where(strikes % 100 == 0, 2000, 0)
    base += round_bonus

    # Add noise
    base += np.random.uniform(200, 800, len(strikes))
    return np.maximum(base, 100).astype(int)

call_oi = synthetic_oi(strikes, spot, 'call')
put_oi = synthetic_oi(strikes, spot, 'put')

# Compute GEX
spot_range = np.linspace(spot * 0.90, spot * 1.10, 500)
gex = compute_gex_profile(spot, strikes, call_oi, put_oi, T, r, sigma, spot_range)

# --- Visualization ---
fig = plt.figure(figsize=(20, 12))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

# Panel 1: Open Interest distribution
ax1 = fig.add_subplot(gs[0, 0])
width = (strikes[1] - strikes[0]) * 0.35
ax1.bar(strikes - width/2, call_oi, width=width, color=CALL_COLOR, alpha=0.7, label='Call OI')
ax1.bar(strikes + width/2, put_oi, width=width, color=PUT_COLOR, alpha=0.7, label='Put OI')
ax1.axvline(spot, color='black', linewidth=1.5, linestyle='--', label=f'Spot = {spot}')
ax1.set_xlabel('Strike')
ax1.set_ylabel('Open Interest (contracts)')
ax1.set_title('Options Open Interest')
ax1.legend(fontsize=9)

# Panel 2: Gamma per strike
ax2 = fig.add_subplot(gs[0, 1])
bs_chain = BlackScholes(S=spot, K=strikes, T=T, r=r, sigma=sigma)
ax2.plot(strikes, bs_chain.gamma(), 'k-', linewidth=2)
ax2.axvline(spot, color='black', linewidth=1.5, linestyle='--')
ax2.set_xlabel('Strike')
ax2.set_ylabel('BS Gamma')
ax2.set_title('Gamma by Strike (peaks ATM)')
ax2.annotate('Gamma peaks at\nthe money', xy=(spot, bs_chain.gamma()[len(strikes)//2]),
             xytext=(spot + 200, bs_chain.gamma().max() * 0.7), fontsize=10,
             arrowprops=dict(arrowstyle='->', color='black'))

# Panel 3: Per-strike GEX breakdown
ax3 = fig.add_subplot(gs[0, 2])
ax3.bar(strikes - width/2, gex['gex_call'] / 1e6, width=width,
        color=LONG_GAMMA_COLOR, alpha=0.7, label='Call GEX (+)')
ax3.bar(strikes + width/2, gex['gex_put'] / 1e6, width=width,
        color=SHORT_GAMMA_COLOR, alpha=0.7, label='Put GEX (−)')
ax3.axvline(spot, color='black', linewidth=1.5, linestyle='--')
ax3.axhline(0, color='black', linewidth=0.5)
ax3.set_xlabel('Strike')
ax3.set_ylabel('GEX ($M per 1% move)')
ax3.set_title('Per-Strike GEX Decomposition')
ax3.legend(fontsize=9)

# Panel 4: Net GEX per strike
ax4 = fig.add_subplot(gs[1, 0])
colors = [LONG_GAMMA_COLOR if g >= 0 else SHORT_GAMMA_COLOR for g in gex['gex_net']]
ax4.bar(strikes, gex['gex_net'] / 1e6, width=width * 2, color=colors, alpha=0.7)
ax4.axvline(spot, color='black', linewidth=1.5, linestyle='--')
ax4.axhline(0, color='black', linewidth=0.5)
ax4.set_xlabel('Strike')
ax4.set_ylabel('Net GEX ($M per 1% move)')
ax4.set_title('Net GEX by Strike')

# Panel 5: Total GEX as function of spot
ax5 = fig.add_subplot(gs[1, 1:])
total = gex['total_gex_by_spot'] / 1e9
ax5.plot(spot_range, total, 'k-', linewidth=2)
ax5.fill_between(spot_range, 0, total,
                 where=(total >= 0), color=LONG_GAMMA_COLOR, alpha=0.3, label='Positive GEX (stabilizing)')
ax5.fill_between(spot_range, 0, total,
                 where=(total < 0), color=SHORT_GAMMA_COLOR, alpha=0.3, label='Negative GEX (destabilizing)')
ax5.axhline(0, color='black', linewidth=1)
ax5.axvline(spot, color='black', linewidth=1.5, linestyle='--', label=f'Current spot = {spot}')

# Find and mark the flip point
sign_changes = np.where(np.diff(np.sign(total)))[0]
for sc in sign_changes:
    flip = spot_range[sc]
    ax5.axvline(flip, color='orange', linewidth=2, linestyle=':', label=f'Gamma flip ≈ {flip:.0f}')
    ax5.plot(flip, 0, 'o', color='orange', markersize=12, zorder=5)

ax5.set_xlabel('Spot Price', fontsize=12)
ax5.set_ylabel('Total GEX ($B per 1% move)', fontsize=12)
ax5.set_title('Total GEX vs Spot Price — The Gamma Landscape', fontsize=13, fontweight='bold')
ax5.legend(fontsize=9, loc='upper left')

plt.suptitle(f'GEX Analysis — Synthetic SPX Chain (T={T*365:.0f} DTE, σ={sigma:.0%})',
             fontsize=16, fontweight='bold', y=1.02)
plt.show()

# Print summary
current_gex = np.sum(gex['gex_net'])
print(f"\nGEX Summary at Spot = {spot}:")
print(f"  Total Call GEX: +${np.sum(gex['gex_call'])/1e9:.2f}B")
print(f"  Total Put GEX:  ${np.sum(gex['gex_put'])/1e9:.2f}B")
print(f"  Net GEX:         ${current_gex/1e9:.2f}B per 1% move")
print(f"  Regime:          {'POSITIVE γ (stabilizing)' if current_gex > 0 else 'NEGATIVE γ (destabilizing)'}")
if len(sign_changes) > 0:
    print(f"  Gamma flip at:   ≈ {spot_range[sign_changes[0]]:.0f}")

---
## 6. The Gamma Flip Point <a id='6-gamma-flip'></a>

The **gamma flip point** (or **zero-gamma level**) is the spot price where aggregate dealer gamma crosses from positive to negative:

$$\text{GEX}(S^*) = 0$$

This level matters because it divides the market into two distinct behavioral regimes:

- **Above the flip:** Positive GEX → mean-reverting, low realized vol, tight ranges.
- **Below the flip:** Negative GEX → momentum-driven, high realized vol, large moves.

### What Moves the Flip Point?

The flip depends on the *balance* between call and put OI. Changes in positioning shift it:

- More put buying → flip moves higher (more of the surface becomes negative gamma)
- More call writing → flip moves lower (positive gamma expands)
- Expiration of near-ATM contracts → can shift the flip significantly

In [ ]:
def find_flip_points(spot_range, total_gex):
    """Find spot levels where GEX crosses zero."""
    flips = []
    for i in range(len(total_gex) - 1):
        if total_gex[i] * total_gex[i + 1] < 0:  # sign change
            # Linear interpolation for precision
            frac = abs(total_gex[i]) / (abs(total_gex[i]) + abs(total_gex[i + 1]))
            flip = spot_range[i] + frac * (spot_range[i + 1] - spot_range[i])
            flips.append(flip)
    return flips

# Show how OI changes shift the flip point
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
spot_range = np.linspace(spot * 0.88, spot * 1.12, 500)

scenarios = [
    ('Baseline', call_oi, put_oi),
    ('Heavy Put Buying\n(+50% put OI)', call_oi, (put_oi * 1.5).astype(int)),
    ('Heavy Call Writing\n(+50% call OI)', (call_oi * 1.5).astype(int), put_oi),
]

for ax, (title, c_oi, p_oi) in zip(axes, scenarios):
    gex_s = compute_gex_profile(spot, strikes, c_oi, p_oi, T, r, sigma, spot_range)
    total = gex_s['total_gex_by_spot'] / 1e9

    ax.plot(spot_range, total, 'k-', linewidth=2)
    ax.fill_between(spot_range, 0, total,
                    where=(total >= 0), color=LONG_GAMMA_COLOR, alpha=0.3)
    ax.fill_between(spot_range, 0, total,
                    where=(total < 0), color=SHORT_GAMMA_COLOR, alpha=0.3)
    ax.axhline(0, color='black', linewidth=1)
    ax.axvline(spot, color='gray', linewidth=1, linestyle='--', alpha=0.5)

    flips = find_flip_points(spot_range, total)
    for f in flips:
        ax.axvline(f, color='orange', linewidth=2, linestyle=':')
        ax.plot(f, 0, 'o', color='orange', markersize=10, zorder=5)
        ax.annotate(f'Flip: {f:.0f}', xy=(f, 0), xytext=(f + 50, total.max() * 0.6),
                    fontsize=10, fontweight='bold', color='orange',
                    arrowprops=dict(arrowstyle='->', color='orange'))

    ax.set_xlabel('Spot Price')
    ax.set_ylabel('Total GEX ($B per 1% move)')
    ax.set_title(title, fontsize=12, fontweight='bold')

plt.suptitle('How Positioning Changes Shift the Gamma Flip Point',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Flip Point Sensitivity to Expiry

As options approach expiry, gamma concentrates at the strike. This means the GEX profile becomes *spikier* and the flip point can shift rapidly as near-term contracts decay.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
spot_range = np.linspace(spot * 0.90, spot * 1.10, 500)

dte_values = [60, 30, 14, 7, 3]
cmap = plt.cm.plasma
colors = [cmap(i / len(dte_values)) for i in range(len(dte_values))]

for dte, color in zip(dte_values, colors):
    T_dte = dte / 365
    gex_t = compute_gex_profile(spot, strikes, call_oi, put_oi, T_dte, r, sigma, spot_range)
    total = gex_t['total_gex_by_spot'] / 1e9
    ax.plot(spot_range, total, color=color, linewidth=2, label=f'{dte} DTE')

    flips = find_flip_points(spot_range, total)
    for f in flips:
        ax.plot(f, 0, 'o', color=color, markersize=8, zorder=5)

ax.axhline(0, color='black', linewidth=1)
ax.axvline(spot, color='gray', linewidth=1, linestyle='--', alpha=0.5)
ax.set_xlabel('Spot Price', fontsize=12)
ax.set_ylabel('Total GEX ($B per 1% move)', fontsize=12)
ax.set_title('GEX Profile by Days to Expiry — Gamma Concentrates Near Expiry',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.annotate('Near expiry: GEX becomes\nspikier and more sensitive\nto spot location',
            xy=(spot - 50, 0), xytext=(spot - 400, 2), fontsize=11,
            arrowprops=dict(arrowstyle='->', color='black'))
plt.tight_layout()
plt.show()

---
## 7. GEX Regimes: Positive vs Negative Gamma <a id='7-gex-regimes'></a>

### The Volatility Suppression Effect

When dealers are net long gamma (positive GEX), their hedging creates a **negative feedback loop**:

1. Price rises → dealers sell stock → selling pressure resists the move
2. Price falls → dealers buy stock → buying pressure resists the move
3. Result: price oscillates in a tighter range → **realized vol < implied vol**

### The Volatility Amplification Effect

When dealers are net short gamma (negative GEX), their hedging creates a **positive feedback loop**:

1. Price rises → dealers buy stock → buying pressure accelerates the move
2. Price falls → dealers sell stock → selling pressure accelerates the move
3. Result: moves are exaggerated → **realized vol > implied vol**

### Quantifying the Impact

The magnitude of dealer flow relative to total market volume determines how much hedging moves the market. The larger the GEX relative to average daily volume, the stronger the regime effect.

We can define a rough **dealer impact ratio**:

$$\text{Impact} \approx \frac{|\text{GEX}| \times \text{expected daily move}}{\text{ADV} \times \text{avg price}}$$

In [ ]:
# Simulate price paths under different gamma regimes
np.random.seed(42)

def simulate_with_gamma_feedback(S0, mu, sigma, T, n_steps, n_paths, gamma_impact):
    """Simulate GBM with a gamma feedback term.

    gamma_impact > 0: dampening (long gamma regime)
    gamma_impact < 0: amplifying (short gamma regime)
    gamma_impact = 0: no feedback (pure GBM)
    """
    dt = T / n_steps
    paths = np.zeros((n_paths, n_steps + 1))
    paths[:, 0] = S0

    for t in range(n_steps):
        dW = np.random.normal(0, np.sqrt(dt), n_paths)
        # Raw return
        raw_return = mu * dt + sigma * dW
        # Gamma feedback: opposes the move if positive, amplifies if negative
        feedback = -gamma_impact * raw_return
        effective_return = raw_return + feedback
        paths[:, t + 1] = paths[:, t] * np.exp(effective_return - 0.5 * sigma**2 * dt)

    return paths

n_paths = 1000
n_steps = 252
T_sim = 1.0

paths_neutral = simulate_with_gamma_feedback(100, 0.0, 0.20, T_sim, n_steps, n_paths, 0.0)
paths_long_gamma = simulate_with_gamma_feedback(100, 0.0, 0.20, T_sim, n_steps, n_paths, 0.3)
paths_short_gamma = simulate_with_gamma_feedback(100, 0.0, 0.20, T_sim, n_steps, n_paths, -0.3)

fig, axes = plt.subplots(2, 3, figsize=(20, 10))

configs = [
    ('No Gamma Effect\n(Pure GBM)', paths_neutral, NEUTRAL_COLOR),
    ('POSITIVE GEX\n(Long Gamma Dealers)', paths_long_gamma, LONG_GAMMA_COLOR),
    ('NEGATIVE GEX\n(Short Gamma Dealers)', paths_short_gamma, SHORT_GAMMA_COLOR),
]

for col, (title, paths, color) in enumerate(configs):
    # Top row: sample paths
    ax = axes[0, col]
    for i in range(min(20, n_paths)):
        ax.plot(paths[i], alpha=0.3, linewidth=0.5, color=color)
    ax.plot(np.mean(paths, axis=0), 'k-', linewidth=1.5, label='Mean')
    ax.axhline(100, color='gray', linewidth=0.5, linestyle=':')
    ax.set_ylim(60, 160)
    ax.set_xlabel('Trading Day')
    ax.set_ylabel('Price')
    ax.set_title(title, fontsize=12, fontweight='bold', color=color)

    # Bottom row: distribution of daily returns
    ax = axes[1, col]
    daily_returns = np.diff(np.log(paths), axis=1).flatten() * 100
    ax.hist(daily_returns, bins=100, density=True, alpha=0.7, color=color, edgecolor='none')
    rvol = np.std(daily_returns)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_xlabel('Daily Return (%)')
    ax.set_ylabel('Density')
    ax.set_title(f'Daily Returns — Realized Vol: {rvol:.2f}%/day', fontsize=11)
    ax.set_xlim(-4, 4)

plt.suptitle('Gamma Regime Effect on Price Dynamics',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Compute realized vols
for name, paths in [('No feedback', paths_neutral), ('Long gamma', paths_long_gamma), ('Short gamma', paths_short_gamma)]:
    daily_rets = np.diff(np.log(paths), axis=1)
    rv_ann = np.std(daily_rets) * np.sqrt(252) * 100
    print(f"{name:15s}: Realized Vol = {rv_ann:.1f}% annualized")

---
## 8. Simulating Market Impact of Dealer Hedging <a id='8-market-impact'></a>

Let's build a more realistic simulation that models dealer hedging as explicit share trades against the market's natural order flow. This lets us see the price impact quantitatively.

In [ ]:
def simulate_dealer_hedging(spot_0, strikes, call_oi, put_oi, T, r, sigma,
                            n_days, market_impact_bps=0.1, adv_shares=5e8, seed=42):
    """Simulate daily prices with explicit dealer hedging flows.

    market_impact_bps: price impact per 1% of ADV traded
    adv_shares: average daily volume in shares
    """
    np.random.seed(seed)
    dt = 1 / 252

    prices = [spot_0]
    dealer_deltas = []  # track total dealer delta position
    hedge_flows = []    # track daily rebalancing flows
    gex_values = []     # track GEX at each step

    # Initial dealer hedge
    prev_delta = 0.0
    for K_i, c_oi, p_oi in zip(strikes, call_oi, put_oi):
        T_rem = T
        bs = BlackScholes(S=spot_0, K=K_i, T=T_rem, r=r, sigma=sigma)
        # Long calls: delta = +N(d1) per contract × 100 shares × OI
        prev_delta += bs.call_delta() * c_oi * 100
        # Short puts: delta = -(N(d1)-1) × OI × 100 = (1-N(d1)) × OI × 100
        prev_delta += (-bs.put_delta()) * p_oi * 100

    for day in range(n_days):
        S = prices[-1]
        T_rem = max(T - day * dt, dt)

        # Natural market move (exogenous)
        natural_return = np.random.normal(0, sigma * np.sqrt(dt))
        S_new_natural = S * np.exp(natural_return)

        # Compute new dealer delta at the naturally moved price
        new_delta = 0.0
        total_gex = 0.0
        for K_i, c_oi, p_oi in zip(strikes, call_oi, put_oi):
            bs = BlackScholes(S=S_new_natural, K=K_i, T=T_rem, r=r, sigma=sigma)
            new_delta += bs.call_delta() * c_oi * 100
            new_delta += (-bs.put_delta()) * p_oi * 100
            g = bs.gamma()
            total_gex += (g * c_oi - g * p_oi) * 100 * S_new_natural**2

        # Hedge flow = change in required stock position
        hedge_shares = new_delta - prev_delta
        # To hedge LONG delta increase → dealer must SHORT stock → selling pressure
        # So the market impact is negative when hedge_shares > 0 (dealer sells)
        flow_as_pct_adv = abs(hedge_shares) / adv_shares
        impact = -np.sign(hedge_shares) * flow_as_pct_adv * market_impact_bps / 10000

        # Final price includes natural move + dealer impact
        S_final = S_new_natural * (1 + impact)

        prices.append(S_final)
        dealer_deltas.append(new_delta)
        hedge_flows.append(-hedge_shares)  # negative = selling
        gex_values.append(total_gex)
        prev_delta = new_delta

    return np.array(prices), np.array(dealer_deltas), np.array(hedge_flows), np.array(gex_values)


# Run simulation
n_days = 60
prices, deltas, flows, gex_ts = simulate_dealer_hedging(
    spot, strikes, call_oi, put_oi, T=90/365, r=r, sigma=sigma,
    n_days=n_days, market_impact_bps=0.5
)

fig, axes = plt.subplots(4, 1, figsize=(16, 16), sharex=True)
days = np.arange(n_days + 1)

# Price
axes[0].plot(days, prices, 'k-', linewidth=1.5)
axes[0].set_ylabel('SPX Price')
axes[0].set_title('Simulated SPX with Dealer Hedging Impact', fontweight='bold')
axes[0].axhline(spot, color='gray', linestyle=':', alpha=0.5)

# GEX over time
gex_bn = gex_ts / 1e9
gex_colors = [LONG_GAMMA_COLOR if g >= 0 else SHORT_GAMMA_COLOR for g in gex_bn]
axes[1].bar(days[1:], gex_bn, color=gex_colors, alpha=0.7, width=0.8)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_ylabel('GEX ($B per 1%)')
axes[1].set_title('Net GEX Over Time')

# Hedge flows
flow_colors = [LONG_GAMMA_COLOR if f >= 0 else SHORT_GAMMA_COLOR for f in flows]
axes[2].bar(days[1:], flows / 1e6, color=flow_colors, alpha=0.7, width=0.8)
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_ylabel('Dealer Flow (M shares)')
axes[2].set_title('Daily Dealer Hedge Rebalancing (+ = buying, − = selling)')

# Cumulative flows
cum_flows = np.cumsum(flows) / 1e6
axes[3].plot(days[1:], cum_flows, color=CALL_COLOR, linewidth=2)
axes[3].fill_between(days[1:], 0, cum_flows, alpha=0.15, color=CALL_COLOR)
axes[3].axhline(0, color='black', linewidth=0.5)
axes[3].set_ylabel('Cumulative Flow (M shares)')
axes[3].set_xlabel('Trading Day')
axes[3].set_title('Cumulative Dealer Hedge Flows')

plt.tight_layout()
plt.show()

---
## 9. GEX and Realized Volatility <a id='9-gex-and-rvol'></a>

The core empirical prediction of the GEX framework: **realized volatility is lower when GEX is positive and higher when GEX is negative.** Let's test this with a Monte Carlo experiment.

In [ ]:
# Monte Carlo: run many paths, bin by GEX regime, compare realized vol
np.random.seed(123)

n_experiments = 200
n_days_exp = 30
results = []

for exp in range(n_experiments):
    # Randomize OI to create different gamma regimes
    oi_scale_call = np.random.uniform(0.3, 2.0)
    oi_scale_put = np.random.uniform(0.3, 2.0)

    c_oi = (call_oi * oi_scale_call).astype(int)
    p_oi = (put_oi * oi_scale_put).astype(int)

    prices_exp, _, _, gex_exp = simulate_dealer_hedging(
        spot, strikes, c_oi, p_oi, T=60/365, r=r, sigma=sigma,
        n_days=n_days_exp, market_impact_bps=0.5, seed=exp
    )

    # Compute realized vol
    daily_rets = np.diff(np.log(prices_exp))
    rvol = np.std(daily_rets) * np.sqrt(252) * 100

    # Average GEX over the period
    avg_gex = np.mean(gex_exp) / 1e9

    results.append({'avg_gex': avg_gex, 'rvol': rvol, 'call_scale': oi_scale_call, 'put_scale': oi_scale_put})

avg_gex_arr = np.array([r['avg_gex'] for r in results])
rvol_arr = np.array([r['rvol'] for r in results])

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Scatter: GEX vs realized vol
ax = axes[0]
colors = [LONG_GAMMA_COLOR if g >= 0 else SHORT_GAMMA_COLOR for g in avg_gex_arr]
ax.scatter(avg_gex_arr, rvol_arr, c=colors, alpha=0.5, s=30, edgecolors='none')

# Trend line
z = np.polyfit(avg_gex_arr, rvol_arr, 1)
p = np.poly1d(z)
x_line = np.linspace(avg_gex_arr.min(), avg_gex_arr.max(), 100)
ax.plot(x_line, p(x_line), 'k--', linewidth=2, label=f'Trend (slope: {z[0]:.2f})')
ax.axvline(0, color='orange', linewidth=1.5, linestyle=':')
ax.set_xlabel('Average GEX ($B per 1% move)', fontsize=11)
ax.set_ylabel('Realized Volatility (% ann.)', fontsize=11)
ax.set_title('GEX vs Realized Volatility', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

# Box plot by regime
ax = axes[1]
pos_gex_rvol = rvol_arr[avg_gex_arr >= 0]
neg_gex_rvol = rvol_arr[avg_gex_arr < 0]
bp = ax.boxplot([pos_gex_rvol, neg_gex_rvol],
                labels=['Positive GEX\n(stabilizing)', 'Negative GEX\n(destabilizing)'],
                patch_artist=True, widths=0.5)
bp['boxes'][0].set_facecolor(LONG_GAMMA_COLOR)
bp['boxes'][1].set_facecolor(SHORT_GAMMA_COLOR)
for box in bp['boxes']:
    box.set_alpha(0.5)
ax.set_ylabel('Realized Volatility (% ann.)', fontsize=11)
ax.set_title('RVol by Gamma Regime', fontsize=13, fontweight='bold')

mean_pos = np.mean(pos_gex_rvol)
mean_neg = np.mean(neg_gex_rvol)
ax.text(1, mean_pos + 1, f'Mean: {mean_pos:.1f}%', ha='center', fontsize=11, color=LONG_GAMMA_COLOR, fontweight='bold')
ax.text(2, mean_neg + 1, f'Mean: {mean_neg:.1f}%', ha='center', fontsize=11, color=SHORT_GAMMA_COLOR, fontweight='bold')

# Histogram
ax = axes[2]
bins = np.linspace(rvol_arr.min(), rvol_arr.max(), 25)
ax.hist(pos_gex_rvol, bins=bins, alpha=0.6, color=LONG_GAMMA_COLOR, label='Positive GEX', density=True)
ax.hist(neg_gex_rvol, bins=bins, alpha=0.6, color=SHORT_GAMMA_COLOR, label='Negative GEX', density=True)
ax.set_xlabel('Realized Volatility (% ann.)', fontsize=11)
ax.set_ylabel('Density')
ax.set_title('RVol Distribution by Regime', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

plt.suptitle('The GEX–Volatility Relationship (Monte Carlo, N=200 experiments)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nResults across {n_experiments} experiments:")
print(f"  Positive GEX regime: mean RVol = {mean_pos:.1f}% (n={len(pos_gex_rvol)})")
print(f"  Negative GEX regime: mean RVol = {mean_neg:.1f}% (n={len(neg_gex_rvol)})")
print(f"  Difference: {mean_neg - mean_pos:.1f} percentage points higher in negative gamma")

---
## 10. Practical Trading Applications <a id='10-practical'></a>

### Application 1: Identifying Support and Resistance from GEX

Strikes with large positive GEX act as "walls" — dealer hedging resists price movement through them. Large negative GEX strikes can act as accelerators.

In [ ]:
# Build a GEX-based support/resistance map
fig, axes = plt.subplots(2, 1, figsize=(16, 12), gridspec_kw={'height_ratios': [2, 1]})

# Recompute with finer detail
gex_detail = compute_gex_profile(spot, strikes, call_oi, put_oi, T, r, sigma)

# Top panel: GEX by strike as horizontal bars (price on Y axis)
ax = axes[0]
gex_net_m = gex_detail['gex_net'] / 1e6

bar_colors = [LONG_GAMMA_COLOR if g >= 0 else SHORT_GAMMA_COLOR for g in gex_net_m]
ax.barh(strikes, gex_net_m, height=20, color=bar_colors, alpha=0.6, edgecolor='none')
ax.axvline(0, color='black', linewidth=0.5)
ax.axhline(spot, color='black', linewidth=2, linestyle='--', label=f'Spot: {spot}')

# Mark top 3 positive GEX (support/resistance)
sorted_idx = np.argsort(gex_net_m)[::-1]
for rank, idx in enumerate(sorted_idx[:3]):
    if gex_net_m[idx] > 0:
        ax.annotate(f'#{rank+1} Wall: {strikes[idx]:.0f}',
                    xy=(gex_net_m[idx], strikes[idx]),
                    xytext=(gex_net_m[idx] + 30, strikes[idx]),
                    fontsize=10, fontweight='bold', color=LONG_GAMMA_COLOR)

# Mark top negative GEX (accelerators)
for rank, idx in enumerate(sorted_idx[-2:]):
    if gex_net_m[idx] < 0:
        ax.annotate(f'Accelerator: {strikes[idx]:.0f}',
                    xy=(gex_net_m[idx], strikes[idx]),
                    xytext=(gex_net_m[idx] - 80, strikes[idx]),
                    fontsize=10, fontweight='bold', color=SHORT_GAMMA_COLOR)

ax.set_ylabel('Strike Price', fontsize=12)
ax.set_xlabel('GEX ($M per 1% move)', fontsize=12)
ax.set_title('GEX Support/Resistance Map', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(spot * 0.92, spot * 1.08)

# Bottom panel: Total GEX curve
ax2 = axes[1]
spot_range_detail = np.linspace(spot * 0.92, spot * 1.08, 300)
gex_curve = compute_gex_profile(spot, strikes, call_oi, put_oi, T, r, sigma, spot_range_detail)
total_curve = gex_curve['total_gex_by_spot'] / 1e9

ax2.plot(spot_range_detail, total_curve, 'k-', linewidth=2)
ax2.fill_between(spot_range_detail, 0, total_curve,
                 where=(total_curve >= 0), color=LONG_GAMMA_COLOR, alpha=0.3)
ax2.fill_between(spot_range_detail, 0, total_curve,
                 where=(total_curve < 0), color=SHORT_GAMMA_COLOR, alpha=0.3)
ax2.axhline(0, color='black', linewidth=1)
ax2.axvline(spot, color='black', linewidth=2, linestyle='--')
ax2.set_xlabel('Spot Price', fontsize=12)
ax2.set_ylabel('Total GEX ($B)', fontsize=12)
ax2.set_title('Total GEX Curve — Green Zone = Suppressed Vol, Red Zone = Amplified Vol',
              fontsize=12, fontweight='bold')

flips = find_flip_points(spot_range_detail, total_curve)
for f in flips:
    ax2.axvline(f, color='orange', linewidth=2, linestyle=':')
    ax2.annotate(f'FLIP: {f:.0f}', xy=(f, 0), xytext=(f + 20, total_curve.max() * 0.5),
                 fontsize=11, fontweight='bold', color='orange',
                 arrowprops=dict(arrowstyle='->', color='orange'))

plt.tight_layout()
plt.show()

### Application 2: Expiration Dynamics — Gamma Unwind

As options expire, the gamma associated with them vanishes. If a large share of GEX is concentrated in near-term contracts, expiration can shift the overall gamma regime dramatically — the so-called **gamma unwind** or **gamma unclenching**.

In [ ]:
# Multi-expiry GEX: show how expiration changes the landscape
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
spot_range = np.linspace(spot * 0.92, spot * 1.08, 300)

# Two expiries: front month (heavy OI) and back month (lighter OI)
T_front = 5 / 365   # 5 DTE
T_back = 35 / 365   # 35 DTE

# Front month has 60% of OI, back month 40%
call_oi_front = (call_oi * 0.6).astype(int)
put_oi_front = (put_oi * 0.6).astype(int)
call_oi_back = (call_oi * 0.4).astype(int)
put_oi_back = (put_oi * 0.4).astype(int)

# Before expiry: total of both
gex_front = compute_gex_profile(spot, strikes, call_oi_front, put_oi_front, T_front, r, sigma, spot_range)
gex_back = compute_gex_profile(spot, strikes, call_oi_back, put_oi_back, T_back, r, sigma, spot_range)
total_before = (gex_front['total_gex_by_spot'] + gex_back['total_gex_by_spot']) / 1e9

# After front expiry: only back month remains
total_after = gex_back['total_gex_by_spot'] / 1e9

# Panel 1: Front month contribution
ax = axes[0]
front_total = gex_front['total_gex_by_spot'] / 1e9
ax.plot(spot_range, front_total, color=SHORT_GAMMA_COLOR, linewidth=2, label='Front month (5 DTE)')
ax.plot(spot_range, gex_back['total_gex_by_spot'] / 1e9, color=CALL_COLOR, linewidth=2, label='Back month (35 DTE)')
ax.fill_between(spot_range, 0, front_total, alpha=0.15, color=SHORT_GAMMA_COLOR)
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(spot, color='black', linewidth=1, linestyle='--')
ax.set_xlabel('Spot Price')
ax.set_ylabel('GEX ($B per 1%)')
ax.set_title('GEX by Expiry', fontweight='bold')
ax.legend(fontsize=9)

# Panel 2: Before vs after expiry
ax = axes[1]
ax.plot(spot_range, total_before, 'k-', linewidth=2.5, label='Before OpEx (total)')
ax.plot(spot_range, total_after, '--', color=CALL_COLOR, linewidth=2.5, label='After OpEx (back only)')
ax.fill_between(spot_range, total_before, total_after, alpha=0.15, color='orange', label='Gamma removed')
ax.axhline(0, color='black', linewidth=1)
ax.axvline(spot, color='black', linewidth=1, linestyle='--')
ax.set_xlabel('Spot Price')
ax.set_ylabel('GEX ($B per 1%)')
ax.set_title('GEX Before vs After Expiration', fontweight='bold')
ax.legend(fontsize=9)

# Panel 3: The percentage change in GEX
ax = axes[2]
with np.errstate(divide='ignore', invalid='ignore'):
    pct_change = np.where(np.abs(total_before) > 0.01,
                          (total_after - total_before) / np.abs(total_before) * 100, 0)
ax.plot(spot_range, pct_change, color='orange', linewidth=2)
ax.fill_between(spot_range, 0, pct_change, alpha=0.15, color='orange')
ax.axhline(0, color='black', linewidth=0.5)
ax.axhline(-50, color='gray', linewidth=0.5, linestyle=':')
ax.axvline(spot, color='black', linewidth=1, linestyle='--')
ax.set_xlabel('Spot Price')
ax.set_ylabel('GEX Change (%)')
ax.set_title('GEX Reduction from Expiration', fontweight='bold')
ax.set_ylim(-100, 20)

plt.suptitle('Expiration Dynamics: The Gamma Unwind',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"GEX at spot before OpEx: ${np.interp(spot, spot_range, total_before):.2f}B")
print(f"GEX at spot after OpEx:  ${np.interp(spot, spot_range, total_after):.2f}B")
print(f"GEX removed by expiry:   {np.interp(spot, spot_range, total_before - total_after):.2f}B")
print(f"→ Expect {'higher' if np.interp(spot, spot_range, total_after) < np.interp(spot, spot_range, total_before) else 'lower'} realized vol after expiration")

### Application 3: GEX Dashboard

A combined view of the key GEX metrics for a snapshot in time.

In [ ]:
def gex_dashboard(spot, strikes, call_oi, put_oi, T, r, sigma, title='GEX Dashboard'):
    """Generate a comprehensive GEX analysis dashboard."""
    spot_range = np.linspace(spot * 0.88, spot * 1.12, 500)
    gex = compute_gex_profile(spot, strikes, call_oi, put_oi, T, r, sigma, spot_range)

    total = gex['total_gex_by_spot'] / 1e9
    flips = find_flip_points(spot_range, total)
    current_gex = np.interp(spot, spot_range, total)

    # Key metrics
    max_gex_strike = strikes[np.argmax(np.abs(gex['gex_net']))]
    total_call_gex = np.sum(gex['gex_call']) / 1e9
    total_put_gex = np.sum(gex['gex_put']) / 1e9
    put_call_ratio = np.sum(put_oi) / np.sum(call_oi)

    fig = plt.figure(figsize=(22, 14))
    gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.4, wspace=0.35,
                          height_ratios=[0.3, 1, 1])

    # Row 0: Summary metrics
    ax_summary = fig.add_subplot(gs[0, :])
    ax_summary.axis('off')
    regime = 'POSITIVE GAMMA (Stabilizing)' if current_gex >= 0 else 'NEGATIVE GAMMA (Destabilizing)'
    regime_color = LONG_GAMMA_COLOR if current_gex >= 0 else SHORT_GAMMA_COLOR
    flip_str = f"{flips[0]:.0f}" if flips else "N/A"

    summary_text = (f"Spot: {spot:,.0f}   |   Net GEX: ${current_gex:.2f}B   |   "
                    f"Regime: {regime}   |   Flip: {flip_str}   |   "
                    f"Put/Call OI Ratio: {put_call_ratio:.2f}")
    ax_summary.text(0.5, 0.5, summary_text, transform=ax_summary.transAxes,
                    fontsize=13, ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.8', facecolor=regime_color, alpha=0.15))

    # Row 1, Col 0-1: GEX curve
    ax1 = fig.add_subplot(gs[1, :2])
    ax1.plot(spot_range, total, 'k-', linewidth=2.5)
    ax1.fill_between(spot_range, 0, total, where=(total >= 0),
                     color=LONG_GAMMA_COLOR, alpha=0.3)
    ax1.fill_between(spot_range, 0, total, where=(total < 0),
                     color=SHORT_GAMMA_COLOR, alpha=0.3)
    ax1.axhline(0, color='black', linewidth=1)
    ax1.axvline(spot, color='black', linewidth=2, linestyle='--')
    for f in flips:
        ax1.axvline(f, color='orange', linewidth=2, linestyle=':')
        ax1.plot(f, 0, 'o', color='orange', markersize=12, zorder=5)
    ax1.set_xlabel('Spot Price', fontsize=11)
    ax1.set_ylabel('Total GEX ($B per 1% move)', fontsize=11)
    ax1.set_title('GEX Landscape', fontsize=13, fontweight='bold')

    # Row 1, Col 2-3: Per-strike breakdown
    ax2 = fig.add_subplot(gs[1, 2:])
    width = (strikes[1] - strikes[0]) * 0.35
    ax2.bar(strikes - width/2, gex['gex_call'] / 1e6, width=width,
            color=LONG_GAMMA_COLOR, alpha=0.7, label='Call GEX')
    ax2.bar(strikes + width/2, gex['gex_put'] / 1e6, width=width,
            color=SHORT_GAMMA_COLOR, alpha=0.7, label='Put GEX')
    ax2.axhline(0, color='black', linewidth=0.5)
    ax2.axvline(spot, color='black', linewidth=1.5, linestyle='--')
    ax2.set_xlabel('Strike', fontsize=11)
    ax2.set_ylabel('GEX ($M)', fontsize=11)
    ax2.set_title('Per-Strike GEX', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=9)

    # Row 2, Col 0-1: OI distribution
    ax3 = fig.add_subplot(gs[2, :2])
    ax3.bar(strikes - width/2, call_oi, width=width, color=CALL_COLOR, alpha=0.7, label='Call OI')
    ax3.bar(strikes + width/2, put_oi, width=width, color=PUT_COLOR, alpha=0.7, label='Put OI')
    ax3.axvline(spot, color='black', linewidth=1.5, linestyle='--')
    ax3.set_xlabel('Strike', fontsize=11)
    ax3.set_ylabel('Open Interest', fontsize=11)
    ax3.set_title('Open Interest Distribution', fontsize=13, fontweight='bold')
    ax3.legend(fontsize=9)

    # Row 2, Col 2-3: Net GEX per strike (horizontal)
    ax4 = fig.add_subplot(gs[2, 2:])
    mask_display = (strikes >= spot * 0.92) & (strikes <= spot * 1.08)
    s_disp = strikes[mask_display]
    g_disp = gex['gex_net'][mask_display] / 1e6
    bar_c = [LONG_GAMMA_COLOR if g >= 0 else SHORT_GAMMA_COLOR for g in g_disp]
    ax4.barh(s_disp, g_disp, height=20, color=bar_c, alpha=0.7)
    ax4.axvline(0, color='black', linewidth=0.5)
    ax4.axhline(spot, color='black', linewidth=1.5, linestyle='--')
    ax4.set_xlabel('Net GEX ($M)', fontsize=11)
    ax4.set_ylabel('Strike', fontsize=11)
    ax4.set_title('Net GEX Map (Walls & Magnets)', fontsize=13, fontweight='bold')

    fig.suptitle(title, fontsize=18, fontweight='bold', y=1.01)
    plt.show()


gex_dashboard(spot, strikes, call_oi, put_oi, T, r, sigma,
              title=f'SPX GEX Dashboard — {T*365:.0f} DTE Snapshot')

### Application 4: Sensitivity Analysis — What Drives GEX?

Explore how GEX changes with volatility, time, and positioning.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
spot_range = np.linspace(spot * 0.90, spot * 1.10, 300)

# Panel 1: GEX vs Implied Volatility
ax = axes[0, 0]
for vol_pct in [12, 18, 25, 35]:
    gex_v = compute_gex_profile(spot, strikes, call_oi, put_oi, T, r, vol_pct/100, spot_range)
    ax.plot(spot_range, gex_v['total_gex_by_spot'] / 1e9, label=f'σ = {vol_pct}%', linewidth=2)
ax.axhline(0, color='black', linewidth=1)
ax.axvline(spot, color='gray', linewidth=1, linestyle='--')
ax.set_xlabel('Spot Price')
ax.set_ylabel('GEX ($B per 1%)')
ax.set_title('GEX Sensitivity to Implied Volatility', fontweight='bold')
ax.legend(fontsize=9)

# Panel 2: GEX vs DTE
ax = axes[0, 1]
for dte in [7, 14, 30, 60, 90]:
    gex_t = compute_gex_profile(spot, strikes, call_oi, put_oi, dte/365, r, sigma, spot_range)
    ax.plot(spot_range, gex_t['total_gex_by_spot'] / 1e9, label=f'{dte} DTE', linewidth=2)
ax.axhline(0, color='black', linewidth=1)
ax.axvline(spot, color='gray', linewidth=1, linestyle='--')
ax.set_xlabel('Spot Price')
ax.set_ylabel('GEX ($B per 1%)')
ax.set_title('GEX Sensitivity to Days to Expiry', fontweight='bold')
ax.legend(fontsize=9)

# Panel 3: Flip point vs vol
ax = axes[1, 0]
vol_range = np.linspace(0.08, 0.50, 50)
flip_by_vol = []
for v in vol_range:
    gex_v = compute_gex_profile(spot, strikes, call_oi, put_oi, T, r, v, spot_range)
    fl = find_flip_points(spot_range, gex_v['total_gex_by_spot'])
    flip_by_vol.append(fl[0] if fl else np.nan)

ax.plot(vol_range * 100, flip_by_vol, 'o-', color=CALL_COLOR, markersize=4)
ax.axhline(spot, color='gray', linewidth=1, linestyle='--', label=f'Current spot = {spot}')
ax.set_xlabel('Implied Volatility (%)')
ax.set_ylabel('Gamma Flip Point')
ax.set_title('Flip Point vs Implied Vol', fontweight='bold')
ax.legend(fontsize=9)

# Panel 4: Current GEX vs put/call OI ratio
ax = axes[1, 1]
ratios = np.linspace(0.3, 3.0, 50)
gex_by_ratio = []
for ratio in ratios:
    # Scale put OI to achieve target ratio
    scaled_put_oi = (put_oi * ratio / (np.sum(put_oi) / np.sum(call_oi))).astype(int)
    gex_r = compute_gex_profile(spot, strikes, call_oi, scaled_put_oi, T, r, sigma)
    gex_by_ratio.append(np.sum(gex_r['gex_net']) / 1e9)

gex_by_ratio = np.array(gex_by_ratio)
ax.plot(ratios, gex_by_ratio, 'k-', linewidth=2)
ax.fill_between(ratios, 0, gex_by_ratio,
                where=(gex_by_ratio >= 0), color=LONG_GAMMA_COLOR, alpha=0.3)
ax.fill_between(ratios, 0, gex_by_ratio,
                where=(gex_by_ratio < 0), color=SHORT_GAMMA_COLOR, alpha=0.3)
ax.axhline(0, color='black', linewidth=1)
ax.axvline(np.sum(put_oi) / np.sum(call_oi), color='orange', linewidth=1.5, linestyle=':',
           label=f'Current ratio: {np.sum(put_oi)/np.sum(call_oi):.2f}')
ax.set_xlabel('Put/Call OI Ratio')
ax.set_ylabel('Net GEX ($B per 1%)')
ax.set_title('Net GEX vs Put/Call OI Ratio', fontweight='bold')
ax.legend(fontsize=9)

plt.suptitle('GEX Sensitivity Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Application 5: Intraday GEX Scenario — What Happens If SPX Drops 3%?

Walk through the dealer hedging cascade that occurs during a sharp selloff.

In [ ]:
# Scenario: SPX drops from 4500 to 4365 (3%) in stages
price_path = np.linspace(spot, spot * 0.97, 20)

cumulative_shares_traded = []
gex_at_each_level = []
dealer_delta_at_each_level = []

# Track dealer delta at each price level
for S in price_path:
    total_delta = 0.0
    total_gex = 0.0
    for K_i, c_oi, p_oi in zip(strikes, call_oi, put_oi):
        bs = BlackScholes(S=S, K=K_i, T=T, r=r, sigma=sigma)
        # Long call delta
        total_delta += bs.call_delta() * c_oi * 100
        # Short put delta (= -put_delta × OI × 100)
        total_delta += (-bs.put_delta()) * p_oi * 100
        g = bs.gamma()
        total_gex += (g * c_oi - g * p_oi) * 100 * S**2

    dealer_delta_at_each_level.append(total_delta)
    gex_at_each_level.append(total_gex)

dealer_delta_arr = np.array(dealer_delta_at_each_level)
gex_arr = np.array(gex_at_each_level)

# Shares dealer must trade at each step = change in delta (sell to rebalance)
hedge_trades = -np.diff(dealer_delta_arr)  # negative diff means selling
cum_hedge = np.cumsum(hedge_trades)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Price path
ax = axes[0, 0]
pct_change = (price_path / spot - 1) * 100
ax.plot(pct_change, price_path, 'k-o', markersize=4, linewidth=2)
ax.set_xlabel('Move from Start (%)')
ax.set_ylabel('SPX Level')
ax.set_title('Scenario: 3% SPX Selloff', fontweight='bold')
ax.invert_xaxis()

# GEX at each level
ax = axes[0, 1]
gex_colors = [LONG_GAMMA_COLOR if g >= 0 else SHORT_GAMMA_COLOR for g in gex_arr]
ax.bar(range(len(gex_arr)), gex_arr / 1e9, color=gex_colors, alpha=0.7)
ax.axhline(0, color='black', linewidth=1)
ax.set_xlabel('Step')
ax.set_ylabel('GEX ($B per 1%)')
ax.set_title('GEX Evolves as Spot Drops', fontweight='bold')

# Add labels for regime
for i, g in enumerate(gex_arr / 1e9):
    if i % 5 == 0:
        ax.text(i, g + 0.05 * np.sign(g), f'{price_path[i]:.0f}',
                ha='center', fontsize=8, rotation=45)

# Hedge trades per step
ax = axes[1, 0]
trade_colors = [SHORT_GAMMA_COLOR if t < 0 else LONG_GAMMA_COLOR for t in hedge_trades]
ax.bar(range(len(hedge_trades)), hedge_trades / 1e6, color=trade_colors, alpha=0.7)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Step')
ax.set_ylabel('Shares Traded (Millions)')
ax.set_title('Dealer Hedge Trades Each Step\n(Negative = Selling)', fontweight='bold')

# Cumulative shares sold
ax = axes[1, 1]
ax.plot(range(len(cum_hedge)), cum_hedge / 1e6, 'o-', color=SHORT_GAMMA_COLOR, linewidth=2, markersize=5)
ax.fill_between(range(len(cum_hedge)), 0, cum_hedge / 1e6, alpha=0.15, color=SHORT_GAMMA_COLOR)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Step')
ax.set_ylabel('Cumulative Shares (Millions)')
ax.set_title('Cumulative Dealer Selling Into Decline', fontweight='bold')

# Dual axis for price
ax2 = ax.twinx()
ax2.plot(range(len(cum_hedge)), price_path[1:], 'k--', alpha=0.4, linewidth=1)
ax2.set_ylabel('SPX Price', color='gray')

plt.suptitle(f'Selloff Cascade: SPX {spot} → {price_path[-1]:.0f} ({(price_path[-1]/spot-1)*100:.1f}%)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nScenario Summary:")
print(f"  SPX move: {spot} → {price_path[-1]:.0f} ({(price_path[-1]/spot-1)*100:.1f}%)")
print(f"  Total shares dealers must sell: {cum_hedge[-1]/1e6:.1f}M")
print(f"  GEX at start: ${gex_arr[0]/1e9:.2f}B → at end: ${gex_arr[-1]/1e9:.2f}B")
regime_start = 'Positive (stabilizing)' if gex_arr[0] > 0 else 'Negative (destabilizing)'
regime_end = 'Positive (stabilizing)' if gex_arr[-1] > 0 else 'Negative (destabilizing)'
print(f"  Regime: {regime_start} → {regime_end}")

---
## 11. Limitations and Extensions <a id='11-limitations'></a>

### Known Limitations

**1. The Positioning Assumption Is a Simplification**

The standard GEX formula assumes customers are systematically long puts and short calls. In practice:
- At some strikes, retail investors aggressively buy calls (e.g., meme stock frenzies).
- Some puts are customer-written (e.g., cash-secured put selling strategies).
- Dealer positioning can vary intraday and across tenors.

More accurate approaches use CBOE volume data or clearing-level transaction records to estimate actual dealer positioning.

**2. Multi-Expiry Complexity**

Real GEX must aggregate across all active expirations. Weekly options, which now dominate SPX volume, create rapidly changing gamma profiles. The simple single-expiry model understates this complexity.

**3. Volatility Surface Effects**

We used a flat volatility surface. In reality, skew and term structure mean each strike has a different IV, which affects gamma calculations. A production implementation would use the full volatility surface.

**4. Discrete Hedging and Transaction Costs**

Dealers don't hedge continuously — they rebalance periodically, creating discrete bursts of flow. Transaction costs also mean dealers tolerate some delta drift before rebalancing.

**5. Market Impact Is Non-Linear**

Our simulation used linear market impact. In reality, large orders have concave impact (diminishing marginal effect), and the price impact of dealer flows depends on market depth, which itself varies.

### Extensions

- **Vanna exposure (VEX):** Sensitivity of delta to implied vol. When vol rises, put deltas increase in magnitude, creating additional selling pressure.
- **Charm exposure:** Delta decay over time (dΔ/dt). As options approach expiry, their deltas change, forcing dealers to rebalance even without price movement.
- **GEX by sector/single-name:** The framework extends beyond indices to individual stocks, where dealer positioning can be even more skewed.
- **Real-time GEX:** Using live options data feeds to compute intraday GEX updates.

In [ ]:
# Quick demonstration of Vanna — the second-order cross-Greek
# Vanna = dΔ/dσ = dν/dS — links vol changes to delta changes

def vanna(S, K, T, r, sigma):
    """Vanna: dDelta/dSigma = dVega/dSpot."""
    bs = BlackScholes(S=S, K=K, T=T, r=r, sigma=sigma)
    sqrt_T = np.sqrt(np.maximum(T, 1e-10))
    return -norm.pdf(bs.d1) * bs.d2 / sigma


fig, axes = plt.subplots(1, 2, figsize=(16, 6))
S_range = np.linspace(spot * 0.85, spot * 1.15, 500)

# Vanna profile
ax = axes[0]
v = vanna(S_range, spot, T, r, sigma)
ax.plot(S_range, v, 'k-', linewidth=2)
ax.fill_between(S_range, 0, v, where=(v >= 0), alpha=0.2, color=LONG_GAMMA_COLOR)
ax.fill_between(S_range, 0, v, where=(v < 0), alpha=0.2, color=SHORT_GAMMA_COLOR)
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(spot, color='gray', linewidth=1, linestyle='--')
ax.set_xlabel('Spot Price')
ax.set_ylabel('Vanna')
ax.set_title('Vanna Profile — dΔ/dσ', fontweight='bold')
ax.annotate('Vol↑ + Below strike:\nDelta shifts more negative\n→ dealers must sell',
            xy=(spot * 0.92, v[100]), xytext=(spot * 0.87, 0.5),
            fontsize=10, arrowprops=dict(arrowstyle='->', color='black'))

# Charm profile (dDelta/dT)
ax = axes[1]
dT = 1/365
bs_now = BlackScholes(S=S_range, K=spot, T=T, r=r, sigma=sigma)
bs_later = BlackScholes(S=S_range, K=spot, T=T-dT, r=r, sigma=sigma)
charm = (bs_later.call_delta() - bs_now.call_delta()) / dT

ax.plot(S_range, charm, 'k-', linewidth=2)
ax.fill_between(S_range, 0, charm, where=(charm >= 0), alpha=0.2, color=LONG_GAMMA_COLOR)
ax.fill_between(S_range, 0, charm, where=(charm < 0), alpha=0.2, color=SHORT_GAMMA_COLOR)
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(spot, color='gray', linewidth=1, linestyle='--')
ax.set_xlabel('Spot Price')
ax.set_ylabel('Charm (dΔ/dT per day)')
ax.set_title('Charm Profile — Delta Decay Over Time', fontweight='bold')
ax.annotate('OTM calls: delta decays\ntoward zero overnight\n→ dealers buy back stock',
            xy=(spot * 1.06, charm[350]), xytext=(spot * 1.08, -0.3),
            fontsize=10, arrowprops=dict(arrowstyle='->', color='black'))

plt.suptitle('Beyond GEX: Vanna and Charm — Higher-Order Dealer Flow Effects',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Summary

### The GEX Framework at a Glance

| Concept | Definition | Practical Meaning |
|---------|------------|-------------------|
| **GEX** | $\sum (\Gamma \cdot OI_{\text{call}} - \Gamma \cdot OI_{\text{put}}) \times 100 \times S^2$ | Dollar hedge flow per 1% move |
| **Positive GEX** | Dealers are net long gamma | Sell rallies, buy dips → low vol |
| **Negative GEX** | Dealers are net short gamma | Buy rallies, sell dips → high vol |
| **Gamma flip** | Spot level where GEX = 0 | Regime boundary |
| **GEX wall** | Strike with large positive GEX | Support/resistance level |
| **Gamma unwind** | GEX collapse from expiration | Vol expansion event |

### Key Takeaways

1. **Dealer hedging is mechanical** — it happens regardless of market views, driven purely by risk management.
2. **The sign of aggregate gamma determines the vol regime** — positive GEX suppresses vol, negative GEX amplifies it.
3. **The positioning assumption matters** — the standard calls-are-sold/puts-are-bought assumption captures dominant flows but misses edge cases.
4. **GEX is dynamic** — it changes with spot, time, vol, and open interest. Expiration events can abruptly shift the regime.
5. **Higher-order effects (vanna, charm) add additional systematic flows** that compound or offset the gamma effect.